# Week 1: Quantum Kernel Starter Notebook
**Project**: Quantum Kernels on CLIP/BERT Embeddings

**Goal this week**: Understand what a quantum kernel is and compute one from scratch.

**By the end of this notebook you will:**
- Understand qubits, quantum gates, and circuits
- Know what a quantum feature map does
- Compute a quantum kernel value between two vectors
- Build a small kernel matrix and verify it is valid

Run each cell top to bottom. Read the markdown cells — they explain *why*, not just *what*.

---
## Step 1: Install dependencies
Run this once. Colab may ask to restart — click Restart if it does.

In [ ]:
!pip install pennylane --quiet
!pip install pennylane-lightning --quiet  # faster CPU simulator
print('Done. If prompted to restart runtime, do so now.')

---
## Step 2: Core imports
PennyLane is our quantum computing framework. Think of it like PyTorch but for quantum circuits.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

print(f'PennyLane version: {qml.__version__}')

---
## Step 3: What is a qubit?
A classical bit is 0 or 1. A **qubit** is a vector in 2D complex space:
`|ψ⟩ = α|0⟩ + β|1⟩` where |α|² + |β|² = 1

When you *measure* a qubit you get 0 with probability |α|² or 1 with probability |β|².
Before measurement it can be in **superposition** — both at once.

**Quantum gates** rotate qubits. RY(θ) rotates by angle θ around the Y-axis of the Bloch sphere.
Let's see what different rotation angles look like:

In [ ]:
# visualize a single qubit being rotated by different angles
dev_1q = qml.device('default.qubit', wires=1)

@qml.qnode(dev_1q)
def single_qubit(theta):
    qml.RY(theta, wires=0)   # rotate around Y axis by theta
    return qml.probs(wires=0) # returns [P(0), P(1)]

angles = np.linspace(0, 2 * np.pi, 50)
probs = np.array([single_qubit(a) for a in angles])

plt.figure(figsize=(8, 3))
plt.plot(angles, probs[:, 0], label='P(|0⟩)', color='steelblue')
plt.plot(angles, probs[:, 1], label='P(|1⟩)', color='coral')
plt.xlabel('Rotation angle θ')
plt.ylabel('Measurement probability')
plt.title('Single qubit RY rotation')
plt.legend()
plt.tight_layout()
plt.show()
print('At θ=π/2 the qubit is in equal superposition: P(0)=P(1)=0.5')

---
## Step 4: What is a quantum feature map?
A **feature map** encodes a classical data vector x into a quantum state |ψ(x)⟩.

The simplest approach is **AngleEmbedding**: map each feature to a rotation angle.
For x = [0.3, 1.2, 0.8, 0.5] with 4 qubits:
- Qubit 0 gets rotated by 0.3
- Qubit 1 gets rotated by 1.2
- etc.

With entangling layers (CNOT gates between qubits), you get cross-feature interactions.
This is the **IQP / ZZ feature map** used in the Havlíček 2019 paper.

In [ ]:
N_QUBITS = 4  # we'll use 4 qubits throughout week 1
dev = qml.device('default.qubit', wires=N_QUBITS)

def iqp_feature_map(x, wires):
    """IQP-style feature map: Hadamard + angle encoding + ZZ entanglement."""
    n = len(wires)
    # Layer 1: put all qubits in superposition
    for w in wires:
        qml.Hadamard(wires=w)
    # Layer 2: encode features as Z rotations
    for i, w in enumerate(wires):
        qml.RZ(x[i % len(x)], wires=w)
    # Layer 3: ZZ entanglement (cross-feature interactions)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i % len(x)] * x[(i+1) % len(x)], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

# visualize the circuit
x_example = np.array([0.3, 1.2, 0.8, 0.5])

@qml.qnode(dev)
def show_circuit(x):
    iqp_feature_map(x, wires=range(N_QUBITS))
    return qml.state()

print(qml.draw(show_circuit)(x_example))
print('\nThis is what your quantum feature map looks like as a circuit.')

---
## Step 5: What is a quantum kernel?
The **quantum kernel** between two vectors x₁ and x₂ is:

`K(x₁, x₂) = |⟨ψ(x₂)|ψ(x₁)⟩|²`

This is the *overlap* (inner product) between the two quantum states.
If x₁ and x₂ are identical, overlap = 1.0.
If they are very different, overlap ≈ 0.0.

**How to compute it**: apply feature map for x₁, then apply *inverse* feature map for x₂,
then measure the probability of getting the all-zeros state |00...0⟩.
That probability equals |⟨ψ(x₂)|ψ(x₁)⟩|².

In [ ]:
@qml.qnode(dev)
def kernel_circuit(x1, x2):
    """Compute quantum kernel K(x1, x2) = |<psi(x2)|psi(x1)>|^2."""
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))  # inverse
    return qml.probs(wires=range(N_QUBITS))

def quantum_kernel(x1, x2):
    """Returns the scalar kernel value."""
    return float(kernel_circuit(x1, x2)[0])  # prob of |0000> state

# sanity checks
x1 = np.array([0.3, 1.2, 0.8, 0.5])
x2 = np.array([0.9, 0.4, 1.1, 0.2])

k_same  = quantum_kernel(x1, x1)  # should be 1.0
k_diff  = quantum_kernel(x1, x2)  # should be < 1.0
k_sym   = quantum_kernel(x2, x1)  # should equal k_diff (symmetry)

print(f'K(x1, x1) = {k_same:.4f}  (expected: 1.0)')
print(f'K(x1, x2) = {k_diff:.4f}  (expected: < 1.0)')
print(f'K(x2, x1) = {k_sym:.4f}  (should equal K(x1,x2) — symmetry check)')
print(f'\nSymmetry holds: {np.isclose(k_diff, k_sym, atol=1e-6)}')

---
## Step 6: Build a kernel matrix
For an SVM, we need to compute K(xᵢ, xⱼ) for every pair of training points.
This N×N matrix is the **kernel matrix** (also called the Gram matrix).

A valid kernel matrix must be:
- **Symmetric**: K(xᵢ, xⱼ) = K(xⱼ, xᵢ)
- **Positive semi-definite**: all eigenvalues ≥ 0

PennyLane has a built-in helper that computes this efficiently.

In [ ]:
# create small toy dataset (20 samples, 4 features)
np.random.seed(42)
X_toy = np.random.uniform(0, np.pi, size=(20, N_QUBITS))
y_toy = (X_toy[:, 0] + X_toy[:, 1] > np.pi).astype(int)  # simple rule

print(f'Dataset: {X_toy.shape[0]} samples, {X_toy.shape[1]} features')
print(f'Classes: {np.sum(y_toy==0)} class-0, {np.sum(y_toy==1)} class-1')

# compute kernel matrix — this may take 30–60 seconds for 20 samples
print('\nComputing 20x20 quantum kernel matrix...')
K_matrix = qml.kernels.kernel_matrix(X_toy, X_toy, quantum_kernel)
print(f'Kernel matrix shape: {K_matrix.shape}')

# verify it is a valid kernel
eigenvalues = np.linalg.eigvalsh(K_matrix)
is_symmetric = np.allclose(K_matrix, K_matrix.T, atol=1e-6)
is_psd       = np.all(eigenvalues >= -1e-6)

print(f'\nSymmetric: {is_symmetric}')
print(f'Positive semi-definite: {is_psd}')
print(f'Min eigenvalue: {eigenvalues.min():.6f}  (should be >= 0)')

# visualize it
plt.figure(figsize=(5, 4))
plt.imshow(K_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(label='K(xᵢ, xⱼ)')
plt.title('Quantum kernel matrix (20×20 toy data)')
plt.xlabel('Sample index j')
plt.ylabel('Sample index i')
plt.tight_layout()
plt.show()

---
## Step 7: Train a quantum kernel SVM
Now plug the kernel matrix into scikit-learn's SVM.
We use `kernel='precomputed'` — sklearn will use our matrix directly
instead of computing its own kernel.

In [ ]:
# split our 20-sample toy set into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_toy, y_toy, test_size=0.3, random_state=42
)

print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

# compute train kernel matrix
print('Computing train kernel matrix...')
K_train = qml.kernels.kernel_matrix(X_train, X_train, quantum_kernel)

# compute test kernel matrix  (shape: n_test x n_train)
print('Computing test kernel matrix...')
K_test = qml.kernels.kernel_matrix(X_test, X_train, quantum_kernel)

# train and evaluate quantum kernel SVM
qsvm = SVC(kernel='precomputed', C=1.0)
qsvm.fit(K_train, y_train)
q_acc = accuracy_score(y_test, qsvm.predict(K_test))

# compare with classical RBF SVM on the same data
rbf_svm = SVC(kernel='rbf', C=1.0)
rbf_svm.fit(X_train, y_train)
c_acc = accuracy_score(y_test, rbf_svm.predict(X_test))

print(f'\nQuantum kernel SVM accuracy : {q_acc:.2%}')
print(f'Classical RBF SVM accuracy  : {c_acc:.2%}')
print('\n(Toy data only — do not read too much into these numbers yet!)')

---
## Step 8: Compute time — understanding the bottleneck
The kernel matrix computation is O(N²) circuit evaluations.
200 training samples = 20,000 circuit evaluations. This is the main speed constraint.
Let's measure how long a single kernel evaluation takes on Colab.

In [ ]:
import time

# time a single kernel evaluation
x_a = np.random.uniform(0, np.pi, N_QUBITS)
x_b = np.random.uniform(0, np.pi, N_QUBITS)

t0 = time.time()
for _ in range(10):
    quantum_kernel(x_a, x_b)
t1 = time.time()

single_eval_ms = (t1 - t0) / 10 * 1000

# estimate compute time for different dataset sizes
print(f'Single kernel evaluation: {single_eval_ms:.1f} ms')
print()
print('Estimated kernel matrix compute time:')
for n in [50, 100, 200, 500]:
    total_s = (n * n / 2) * (single_eval_ms / 1000)  # upper triangle only
    unit = 'sec' if total_s < 60 else 'min'
    val = total_s if total_s < 60 else total_s / 60
    print(f'  N={n:4d} samples  ->  ~{val:.1f} {unit}')

print('\nRecommendation: use N=200 for experiments, N=500 only for final runs.')

---
## Week 1 Complete — Summary & Next Steps
### What you learned:
- A **qubit** is a 2D complex vector; gates rotate it; measurement collapses it
- A **feature map** encodes data x as a quantum state |ψ(x)⟩
- A **quantum kernel** K(x₁,x₂) = |⟨ψ(x₂)|ψ(x₁)⟩|² measures state overlap
- A valid kernel matrix is symmetric and positive semi-definite
- Computation time is O(N²) — plan dataset sizes accordingly

### Week 2 preview:
You will swap the toy dataset for real BERT embeddings from SST-2 (sentiment)
or CLIP embeddings from CIFAR-10. The quantum kernel code stays identical —
only the data pipeline changes.

### Reading for this week:
1. Havlíček et al. 2019 — https://arxiv.org/abs/1804.11326
2. PennyLane kernel tutorial — https://pennylane.ai/qml/demos/tutorial_kernel_based_training
3. Schuld & Killoran 2019 — https://arxiv.org/abs/1803.07128